In [ ]:
import numpy as np
import pandas as pd
import sys
import time



In [5]:
import numpy as np
import pandas as pd
import time
import sys

sys.path.append("..")
import metodos_estadisticos as tests

ruta_data = "../data"
ruta_resultados = "../resultados"

# joblib viene incluido con scikit-learn, muy probable que ya lo tengas
from joblib import Parallel, delayed

# --- Función worker ---
def procesar_muestra(escenario, i, x, y):
    import sys
    sys.path.append("..")
    import metodos_estadisticos as tests  # re-importar en cada worker
    import time

    resultados_locales = []
    metodos = [
        ("t_student",      lambda: tests.test_t_student(x, y)),
        ("wilcoxon",       lambda: tests.test_wilcoxon(x, y)),
        ("hodges_lehmann", lambda: tests.test_hodges_lehmann(x, y)),
        ("perm_media",     lambda: tests.test_permutacion(x, y, "media")),
        ("perm_mediana",   lambda: tests.test_permutacion(x, y, "mediana")),
        ("perm_trim",      lambda: tests.test_permutacion(x, y, "trim")),
    ]

    for nombre, fn in metodos:
        t0 = time.time()
        res = int(fn())
        dt = time.time() - t0
        resultados_locales.append({
            "escenario": escenario,
            "muestra": i,
            "metodo": nombre,
            "resultado": res,
            "tiempo": dt
        })

    return resultados_locales


# --- Construcción de tareas ---
tareas = []
for escenario in range(1, 14):
    data = np.load(f"{ruta_data}/datos_escenario_{escenario}.npz")
    xs, ys = data["x"], data["y"]
    for i, (x, y) in enumerate(zip(xs, ys)):
        tareas.append((escenario, i, x, y))

print(f"Total de tareas: {len(tareas)}")

# --- Ejecución paralela con joblib ---
# n_jobs=-1 usa todos los núcleos disponibles, -2 deja uno libre
resultados_anidados = Parallel(n_jobs=-2, verbose=10)(
    delayed(procesar_muestra)(esc, i, x, y)
    for esc, i, x, y in tareas
)

# --- Aplanar lista de listas y guardar ---
resultados = [item for sublista in resultados_anidados for item in sublista]

df = pd.DataFrame(resultados)
df.to_csv(f"{ruta_resultados}/resultados_tests.csv", index=False)
print(f"Listo. {len(df)} registros guardados.")

Total de tareas: 6500


[Parallel(n_jobs=-2)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-2)]: Done   3 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-2)]: Done  10 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-2)]: Done  19 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-2)]: Done  28 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-2)]: Done  39 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-2)]: Done  50 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-2)]: Done  63 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-2)]: Done  76 tasks      | elapsed:    2.0s
[Parallel(n_jobs=-2)]: Done  91 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-2)]: Done 106 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-2)]: Done 123 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-2)]: Done 140 tasks      | elapsed:    3.6s
[Parallel(n_jobs=-2)]: Done 159 tasks      | elapsed:    4.0s
[Parallel(n_jobs=-2)]: Done 178 tasks      | elapsed:    4.5s
[Parallel(n_jobs=-2)]: Done 199 tasks      | elapsed:  

Listo. 39000 registros guardados.


[Parallel(n_jobs=-2)]: Done 6500 out of 6500 | elapsed:  2.6min finished
